## 1. Simulação de Degradação e Detecção de Data Drift

Nesta etapa, o objetivo é simular a passagem do tempo e o desgaste operacional em produção[cite: 6]. Para isso, realizamos as seguintes ações:
1. **Carregamento dos Datasets:** Importação dos dados de referência (baseline de treino) e dos dados de produção atuais.
2. **Injeção Intencional de Drift:** Modificação proposital de variáveis críticas (`Credit amount` e `Duration`) para simular mudanças no comportamento dos clientes e inflação/prazos.
3. **Auditoria Estatística:** Aplicação de testes automatizados para identificar desvios de distribuição entre as bases.

### 1.1 Simulação de Drift e Geração do Relatório

Nesta etapa, carregamos os datasets de referência e produção, injetamos alterações intencionais nas variáveis críticas (`Credit amount` e `Duration`) para simular a degradação dos dados ao longo do tempo, e executamos a suíte de testes do Evidently para gerar e salvar o relatório HTML interativo na pasta de relatórios.

In [1]:
from pathlib import Path
import pandas as pd
from evidently.test_suite import TestSuite
from evidently.test_preset import DataDriftTestPreset

# 1. Carrega os datasets originais
df_reference = pd.read_csv("../data/reference/reference_data.csv")
df_production = pd.read_csv("../data/production/production_data.csv")

# =====================================================================
# SIMULAÇÃO INTENCIONAL DE DRIFT (Exigência da Etapa 2)
# Vamos alterar intencionalmente a distribuição de duas variáveis importantes
# =====================================================================
# Exemplo 1: Inflar artificialmente os valores de uma coluna numérica (ex: Credit amount / Duration)
if 'Credit amount' in df_production.columns:
    df_production['Credit amount'] = df_production['Credit amount'] * 1.85

# Exemplo 2: Modificar intencionalmente outra variável (ex: Age ou Duration)
if 'Duration' in df_production.columns:
    df_production['Duration'] = df_production['Duration'] + 12

print("[SIMULAÇÃO] Dados de produção alterados intencionalmente para gerar drift.")

# 2. Configura a suite de testes de data drift
drift_test_suite = TestSuite(tests=[
    DataDriftTestPreset(),
])

# 3. Executa a análise comparativa com os dados modificados
drift_test_suite.run(reference_data=df_reference, current_data=df_production)

# 4. Salva o relatório HTML interativo na pasta reports/ na raiz
output_dir = Path("../reports")
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / "data_drift_report.html"

drift_test_suite.save_html(str(report_path))
print(f"[SUCESSO] Relatório HTML com drift simulado gerado em: {report_path}")

[SIMULAÇÃO] Dados de produção alterados intencionalmente para gerar drift.
[SUCESSO] Relatório HTML com drift simulado gerado em: ..\reports\data_drift_report.html


### 1.2 Auditoria Detalhada dos Testes de Drift (Evidently AI)

Nesta etapa complementar, extraímos programaticamente o dicionário de resultados da suíte executada para listar no console quais testes específicos falharam e quais colunas apresentaram desvio estatístico relevante, garantindo rastreabilidade direta no notebook.

In [2]:
# 1. Extração detalhada do resumo dos testes de drift da Etapa 2
suite_dict = drift_test_suite.as_dict()

total_tests = len(suite_dict.get("tests", []))
failed_tests_list = [t for t in suite_dict.get("tests", []) if t.get("status") == "FAIL"]

print(f"=== RELATÓRIO DE AUDITORIA DE DRIFT (ETAPA 2) ===")
print(f"Total de testes executados: {total_tests}")
print(f"Testes com sucesso: {total_tests - len(failed_tests_list)}")
print(f"Testes com falha (Drift Detectado): {len(failed_tests_list)}")

if failed_tests_list:
    print("\n[ALERTA] Detalhes dos testes que falharam:")
    for i, test in enumerate(failed_tests_list, 1):
        print(f"  {i}. {test.get('name')}: {test.get('description')}")
else:
    print("\n[OK] Nenhum teste de drift falhou.")

=== RELATÓRIO DE AUDITORIA DE DRIFT (ETAPA 2) ===
Total de testes executados: 11
Testes com sucesso: 9
Testes com falha (Drift Detectado): 2

[ALERTA] Detalhes dos testes que falharam:
  1. Drift per Column: The drift score for the feature **Credit amount** is 0. The drift detection method is K-S p_value. The drift detection threshold is 0.05.
  2. Drift per Column: The drift score for the feature **Duration** is 0. The drift detection method is K-S p_value. The drift detection threshold is 0.05.


### 1.3 Verificação Individual de Desvio nas Variáveis-Chave

Aqui realizamos uma verificação direta nas colunas modificadas (`Credit amount` e `Duration`) para comparar estatísticas descritivas básicas (média e desvio padrão) entre a base de referência e a base de produção modificada.

In [3]:
# 1. Comparação estatística rápida das colunas alteradas na Etapa 2
features_to_check = ['Credit amount', 'Duration']

print("=== COMPARAÇÃO ESTATÍSTICA (REFERÊNCIA vs PRODUÇÃO) ===")
for col in features_to_check:
    if col in df_reference.columns and col in df_production.columns:
        ref_mean = df_reference[col].mean()
        prod_mean = df_production[col].mean()
        diff_pct = ((prod_mean - ref_mean) / ref_mean) * 100
        
        print(f"\nVariável: {col}")
        print(f"  - Média Referência: {ref_mean:.2f}")
        print(f"  - Média Produção:   {prod_mean:.2f}")
        print(f"  - Variação (Drift): {diff_pct:+.2f}%")

=== COMPARAÇÃO ESTATÍSTICA (REFERÊNCIA vs PRODUÇÃO) ===

Variável: Credit amount
  - Média Referência: 3307.67
  - Média Produção:   5894.64
  - Variação (Drift): +78.21%

Variável: Duration
  - Média Referência: 21.07
  - Média Produção:   32.52
  - Variação (Drift): +54.34%
